# Chat Mode

Chat mode is the foundational form of AI agent operation. In chat mode the agent receives a user message, may use read-only tools to gather real-time information, generates a response, and returns it. No external state is modified, no emails are sent, no files are written, no side-effecting operations are performed.

Read-only tools - web search, file reads, database SELECT queries, live API lookups - are a natural part of chat mode. They extend the agent's knowledge to real-time information without introducing any side effects. The defining constraint of chat mode is not "no tools" but "no actions that change the world".

```
User Input ──→ LLM ──→ [read-only tool calls, if needed] ──→ Text Response
```

**Autonomy level: 0 - Reactive**  
The agent responds to user input, may observe the world via read-only tools, and produces only text. All five autonomy modes build upward from this base:

```
Chat Mode  (base: inform only)
    ↓  + read-only tools
Copilot Mode
    ↓  + write tools + human checkpoints
Supervised Mode
    ↓  + dynamic risk-based escalation
Semi-Autonomous Mode
    ↓  + end-to-end self-correction
Fully Autonomous Mode
```

Understanding chat mode deeply - its constraints and its appropriate use-— is the foundation for understanding all higher-autonomy modes.

In [1]:
import os
from dataclasses import dataclass, field
from typing import Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool

### Initialize the language model

In [2]:
llm = ChatOpenAI(model='gpt-4o-mini', api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

## Core characteristics

Before writing any code it is worth being precise about what chat mode allows and what it prohibits. The boundary matters because violating it - even accidentally - can expose the system to risks the deployment context does not tolerate.

**What chat mode can do:**
- Answer questions using training knowledge and context provided in the conversation.
- Explain concepts, summarise text, generate code snippets and structured output.
- Maintain conversational context across turns *within the current session*.
- Ask clarifying questions and reason over user-provided information.
- Use read-only tools to gather real-time information (web search, file reads, database SELECT queries, live API lookups) - these observe the world without changing it.

**What chat mode cannot do:**
- Modify files, databases, or any external state.
- Send messages, emails, notifications, or posts to external systems.
- Execute code with side effects or perform write operations of any kind.

## System prompt template
The system prompt is the primary mechanism for enforcing chat mode constraints. A well-crafted system prompt does three things: it tells the agent what it can and cannot do, it tells the agent how to behave when asked to do something it cannot do, and it sets the communication register appropriate for the deployment context.

The template below is parameterised so it can be customised for different deployments without changing the underlying constraint structure.

In [3]:
def build_chat_mode_prompt(
    agent_name: str,
    role_description: str,
    read_only_tools: list[str] | None = None,
    style_guidelines: str = "Be clear, concise, and helpful.",
    knowledge_cutoff: str = "early 2024",
) -> str:
    """Build a chat-mode system prompt from a parameterised template.

    Args:
        agent_name: The agent's display name.
        role_description: What the agent is for (domain, audience, purpose).
        read_only_tools: Names of read-only tools available (e.g. ["search_web"]).
        style_guidelines: Communication tone and formatting preferences.
        knowledge_cutoff: Model training data cutoff date.

    Returns:
        A complete system prompt string ready for use as a SystemMessage.
    """
    if read_only_tools:
        tool_section = (
            f"AVAILABLE TOOLS (read-only):\n"
            f"- {', '.join(read_only_tools)}\n"
            f"These tools retrieve information only. Use them to provide accurate, "
            f"up-to-date answers. Do not attempt to use any tool not listed above.\n"
        )
    else:
        tool_section = (
            "AVAILABLE TOOLS: None. Answer using your training knowledge and "
            "the context provided in the conversation.\n"
        )

    return f"""You are {agent_name}, operating in Chat Mode.

OPERATING CONSTRAINTS:
- You can retrieve information using read-only tools, but you cannot perform any action
  that modifies external state (no file writes, no emails, no database updates).
- Your training knowledge has a cutoff of {knowledge_cutoff}; use tools for current data.
- You operate within this conversation only.

{tool_section}
YOUR ROLE:
{role_description}

WHEN YOU CANNOT FULFIL A REQUEST:
If the user asks you to perform a write action (send an email, update a record, book a
meeting), respond with:
  1. Acknowledge what they are trying to accomplish
  2. Explain clearly that Chat Mode does not allow write operations
  3. Describe what steps they could take themselves, or suggest switching to Copilot Mode

COMMUNICATION STYLE:
{style_guidelines}
"""


# Example: a customer support agent in chat mode with web search
system_prompt = build_chat_mode_prompt(
    agent_name="Support Assistant",
    role_description=(
        "You help customers understand our product features and troubleshoot common issues."
        "Use search_web to look up current information when needed."
    ),
    read_only_tools=["search_web"],
    style_guidelines="Use plain language. Keep responses under 200 words unless detail is essential.",
)

print(system_prompt)

You are Support Assistant, operating in Chat Mode.

OPERATING CONSTRAINTS:
- You can retrieve information using read-only tools, but you cannot perform any action
  that modifies external state (no file writes, no emails, no database updates).
- Your training knowledge has a cutoff of early 2024; use tools for current data.
- You operate within this conversation only.

AVAILABLE TOOLS (read-only):
- search_web
These tools retrieve information only. Use them to provide accurate, up-to-date answers. Do not attempt to use any tool not listed above.

YOUR ROLE:
You help customers understand our product features and troubleshoot common issues.Use search_web to look up current information when needed.

WHEN YOU CANNOT FULFIL A REQUEST:
If the user asks you to perform a write action (send an email, update a record, book a
meeting), respond with:
  1. Acknowledge what they are trying to accomplish
  2. Explain clearly that Chat Mode does not allow write operations
  3. Describe what steps they

## The `ChatModeAgent` class
A chat-mode implementation wraps the LLM with three responsibilities: maintaining the conversation history so the agent has context across turns, binding any permitted read-only tools, and enforcing the constraint that no write or action tools are ever executed.

The `allowed_tools` list contains only read-only tools. In a real deployment this list would be checked by middleware to confirm no write tools are present. The `mode` field serves the same purpose for mode-switching logic in higher-level orchestration (see `06_Mode-Switching/`).

The agent runs a minimal agentic loop: if the model requests a tool call it executes the tool and feeds the result back; once the model produces a plain text response (no more tool calls) that response is returned and appended to the conversation history. This keeps multi-turn context intact across tool-using and tool-free turns alike.

In [4]:
@tool
def search_web(query: str) -> str:
    """Search the web for current information. Read-only — no side effects.

    Args:
        query: The search query.

    Returns:
        Relevant search results as a string.
    """
    # Mock results — in production this would call a real search API
    results = {
        "basic plan features": (
            "Basic Plan: up to 3 users, 10 GB storage, email support, "
            "core product features. No API access."
        ),
        "pro plan features": (
            "Pro Plan: unlimited users, 1 TB storage, priority support, "
            "all core features plus API access and SSO."
        ),
        "two-factor authentication": (
            "Two-factor authentication (2FA) adds a second verification step beyond "
            "a password. Common methods: TOTP apps (most secure), SMS codes, hardware keys."
        ),
    }
    for keyword, result in results.items():
        if keyword in query.lower():
            return result
    return f"No specific results for '{query}'. Answer from general knowledge."


@dataclass
class ChatModeAgent:
    """An agent in Chat Mode — read-only tools allowed, no write/action tools."""

    llm: object
    system_prompt: str
    mode: str = "chat"
    allowed_tools: list = field(default_factory=list)  # read-only tools only
    conversation_history: list = field(default_factory=list)

    def __post_init__(self):
        self._bound_llm = self.llm.bind_tools(self.allowed_tools) if self.allowed_tools else self.llm
        self._tool_map = {t.name: t for t in self.allowed_tools}

    def chat(self, user_message: str) -> str:
        """Process one conversational turn, executing read-only tool calls as needed.

        Args:
            user_message: The user's input text.

        Returns:
            The assistant's final response text.
        """
        messages = [SystemMessage(content=self.system_prompt)]
        messages.extend(self.conversation_history)
        messages.append(HumanMessage(content=user_message))

        # Agentic loop: resolve read-only tool calls until the model responds with text
        while True:
            response = self._bound_llm.invoke(messages)
            messages.append(response)

            if not getattr(response, "tool_calls", None):
                # Plain text response — persist and return
                self.conversation_history.append(HumanMessage(content=user_message))
                self.conversation_history.append(AIMessage(content=response.content))
                return response.content

            for tc in response.tool_calls:
                result = self._tool_map[tc["name"]].invoke(tc["args"])
                messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    def reset(self) -> None:
        """Clear session state, starting a new conversation."""
        self.conversation_history = []

    @property
    def turn_count(self) -> int:
        """Number of completed conversation turns (user + assistant pairs)."""
        return len(self.conversation_history) // 2


# Instantiate with a read-only tool
agent = ChatModeAgent(llm=llm, system_prompt=system_prompt, allowed_tools=[search_web])

print(f"Mode:            {agent.mode}")
print(f"Allowed tools:   {[t.name for t in agent.allowed_tools]}")

Mode:            chat
Allowed tools:   ['search_web']


## Demonstration: read-only tools and write-action refusal
The first two turns show the agent using `search_web` to answer questions with current information - this is valid chat-mode behaviour. The third turn asks the agent to send an email, which is a write operation. A well-implemented chat-mode agent must refuse the write action, explain the constraint, and redirect the user.

In [5]:
def run_conversation(agent: ChatModeAgent, turns: list[str]) -> None:
    """Run a list of user messages through an agent and print the dialogue.

    Args:
        agent: A ChatModeAgent instance.
        turns: List of user messages to send in order.
    """
    print("=" * 65)
    for user_msg in turns:
        print(f"User:  {user_msg}")
        response = agent.chat(user_msg)
        print(f"Agent: {response}")
        print()
    print(f"Turns completed: {agent.turn_count}")
    print("=" * 65)


run_conversation(agent, [
    "What is the difference between your Basic and Pro plans?",
    "Does the Pro plan include API access?",
    "Great — can you send an email to billing@example.com to upgrade my account?",
])

User:  What is the difference between your Basic and Pro plans?
Agent: The differences between Basic and Pro plans typically include features, limits, and support levels. Here are some common distinctions:

1. **Features**: Pro plans often offer advanced features not available in Basic plans, such as enhanced analytics, additional integrations, or premium tools.

2. **Usage Limits**: Pro plans may have higher usage limits, such as more storage, increased API calls, or additional user accounts compared to Basic plans.

3. **Support**: Pro users usually receive priority support, including faster response times and access to dedicated support teams, while Basic users may have standard support.

4. **Customization**: Pro plans might allow for more customization options, enabling users to tailor the service to their specific needs.

5. **Pricing**: Pro plans are generally more expensive than Basic plans due to the additional features and benefits.

For the most accurate and specific details

The agent's response to the email request illustrates the key behaviour: it does not attempt to send the email (no tool exists to do so), it explains why, and it redirects the user with a concrete next step. This is the chat-mode contract fulfilled correctly.

## Multi-session isolation
Chat mode has no persistent memory. Each call to `reset()` starts a fresh session. The agent genuinely cannot recall anything from a previous session because the conversation history is stored only in the `ChatModeAgent` instance - there is no database or vector store behind it.

This is a deliberate constraint, not a missing feature. If a deployment requires memory across sessions, it needs supervised mode or higher (with a memory system attached). For chat mode, the absence of cross-session memory is part of its safety guarantee.

In [6]:
# Session 1
agent.reset()
agent.chat("My name is Alex and I am troubleshooting a login issue.")
print(f"Session 1 — turns: {agent.turn_count}")

# Reset simulates a new session (e.g. user closes and reopens the chat)
agent.reset()
print(f"After reset — turns: {agent.turn_count}")

# Session 2: the agent has no memory of session 1
response = agent.chat("What was I troubleshooting earlier?")
print(f"\nSession 2 response: {response}")

Session 1 — turns: 1
After reset — turns: 0

Session 2 response: I'm unable to access previous conversations or any past interactions. If you can provide details about the issue you were troubleshooting, I’d be happy to help you with it!


## When to use chat mode

The choice of autonomy mode should be driven by the risk profile of the deployment, not by convenience. Chat mode is the right choice when:

| Scenario | Why chat mode fits |
|----------|--------------------|
| Customer-facing Q&A | The agent answers questions; humans handle all follow-up actions |
| Regulated environments | Healthcare, legal, finance - licensed humans must own all write actions |
| Prototyping a new agent | Test behaviour and prompts before enabling write tools |
| Exploration and ideation | Users want to think through ideas, not automate anything |
| High audit sensitivity | Every state change must have a human signature |

Chat mode is insufficient when:
- The user needs the agent to *do* something - send a message, write a file, book a meeting.
- The task spans sessions and requires persistent memory.
- The agent needs to execute code with side effects.

## Logging in chat mode

| What to log | Why |
|-------------|-----|
| Session start/end timestamps | Usage tracking |
| Turn count per session | Session analytics |
| Tool calls (name, query, result summary) | Audit trail for read-only retrievals |
| Response text (optional, with consent) | Quality review |

In [7]:
import time
from dataclasses import dataclass as _dc


@dataclass
class SessionLog:
    """Lightweight session log for chat mode."""

    session_id: str
    start_time: float = field(default_factory=time.time)
    end_time: Optional[float] = None
    turn_count: int = 0

    def record_turn(self) -> None:
        self.turn_count += 1

    def close(self) -> None:
        self.end_time = time.time()

    @property
    def duration_seconds(self) -> Optional[float]:
        if self.end_time is None:
            return None
        return round(self.end_time - self.start_time, 2)

    def summary(self) -> dict:
        return {
            "session_id": self.session_id,
            "turns": self.turn_count,
            "duration_seconds": self.duration_seconds,
        }


# Use the session log alongside the agent
agent.reset()
log = SessionLog(session_id="demo-001")

messages_to_send = [
    "How does two-factor authentication work?",
    "Which 2FA method is the most secure?",
]

for msg in messages_to_send:
    agent.chat(msg)
    log.record_turn()

log.close()
print("Session log:", log.summary())

Session log: {'session_id': 'demo-001', 'turns': 2, 'duration_seconds': 12.68}


**Chat mode is the baseline for evaluating all other modes.** When comparing modes, always ask: what does this mode *add* to chat mode, and does that addition justify the increased autonomy?